# Fabric Delta Table Maintenance

Single-file Fabric notebook (Runtime 1.3, Spark 3.5, Delta 3.2). Runs OPTIMIZE, V-Order, VACUUM, and optional repartitioning across all Delta tables in a Lakehouse.

## How to use

1. Upload this notebook to your Fabric workspace.
2. Confirm the notebook is on **Runtime 1.3**.
3. Open Cell 2 (parameters) and edit the two ABFSS paths to point at your target Lakehouse and a separate log Lakehouse.
4. First run: keep `dry_run = True`. Validate discovery and planned operations.
5. Real run: set `dry_run = False` (preferably via a Fabric Pipeline parameter override; see the scheduling section at the bottom).

## Cell layout

| Cell | Purpose |
|------|---------|
| 1 | Maintenance module (inlined from `src/fabric_delta_maintenance.py`). |
| 2 | **Parameter cell** (tagged `parameters`). Pipeline-overridable scalars. |
| 3 | Build `MaintenanceConfig` and invoke `run_maintenance`. |
| 4 | Inspect the run summary. |
| 5 | Query the `_maintenance_log` Delta table. |
| 6 | Optional: file-compaction trend analysis. |
| 7 | Scheduling and pipeline parameter walkthrough (markdown). |


In [ ]:
"""
Fabric Delta Table Maintenance Module
=====================================
Runtime: Microsoft Fabric Runtime 1.3 (Spark 3.5, Delta 3.2)
Purpose: Automated OPTIMIZE, V-Order, VACUUM, and conditional repartitioning
         for all Delta tables in a Lakehouse via ABFSS path.

Usage:
    Paste into a Fabric Notebook cell, then call run_maintenance(spark, config).
    The Lakehouse does NOT need to be attached; all operations use ABFSS paths.

Author: Prasanth Sistla
"""

from __future__ import annotations

import json
import logging
import time
import traceback
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    DoubleType, TimestampType, IntegerType, BooleanType
)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

MIN_VACUUM_RETENTION_HOURS = 168  # 7 days, hard floor

# Auto-throttle thresholds (bytes)
LARGE_TABLE_THRESHOLD_BYTES = 10 * 1024**3       # 10 GB
VERY_LARGE_TABLE_THRESHOLD_BYTES = 50 * 1024**3   # 50 GB

# Repartition safety ceiling
DEFAULT_REPARTITION_SIZE_LIMIT_GB = 500.0

# ---------------------------------------------------------------------------
# SKU profiles: guardrails derived from Spark vCore allocations
#
# vCores = CU * 2.  Burst multiplier = 3x for F32+.
#   F32:  64 base /  192 burst vCores
#   F64: 128 base /  384 burst vCores
#   F128: 256 base / 768 burst vCores
#
# max_parallel_workers: safe ceiling before CU contention degrades
#     throughput. A single OPTIMIZE can saturate 40-60 vCores on a
#     multi-GB table, so these are conservative.
# max_repartition_gb: largest table size we allow for the staging
#     rewrite pattern, based on available executor memory.
# optimize_warn_gb: log a warning when OPTIMIZE targets a table
#     larger than this (not a hard block, just observability).
# ---------------------------------------------------------------------------

SUPPORTED_SKUS = {"F32", "F64", "F128", "F256", "F512", "F1024", "F2048"}

# Fabric artifact types whose Delta tables are system-managed and must not be
# touched by user-driven OPTIMIZE / VACUUM / repartitioning. Fabric owns layout,
# compaction, and checkpoint management for these; external writes can corrupt
# the artifact's catalog state. ABFSS path segments take the form
# "{name}.{ArtifactType}" (for example, "sales.Warehouse"), so we match on the
# lowercased ".artifacttype" suffix.
MANAGED_ARTIFACT_SUFFIXES = (
    ".warehouse",
    ".mirroreddatabase",
    ".kqldatabase",
    ".sqldatabase",
)
LAKEHOUSE_ARTIFACT_SUFFIX = ".lakehouse"

# Worker ceilings above F128 grow sublinearly: py4j gateway thread
# serialization dominates past ~16 driver threads regardless of vCores
# (see Known Limitation #3 in CLAUDE.md).
SKU_PROFILES: dict[str, dict] = {
    "F32": {
        "base_vcores": 64,
        "burst_vcores": 192,
        "max_parallel_workers": 2,
        "max_repartition_gb": 50.0,
        "optimize_warn_gb": 100.0,
    },
    "F64": {
        "base_vcores": 128,
        "burst_vcores": 384,
        "max_parallel_workers": 4,
        "max_repartition_gb": 200.0,
        "optimize_warn_gb": 300.0,
    },
    "F128": {
        "base_vcores": 256,
        "burst_vcores": 768,
        "max_parallel_workers": 8,
        "max_repartition_gb": 500.0,
        "optimize_warn_gb": 750.0,
    },
    "F256": {
        "base_vcores": 512,
        "burst_vcores": 1536,
        "max_parallel_workers": 12,
        "max_repartition_gb": 1000.0,
        "optimize_warn_gb": 1500.0,
    },
    "F512": {
        "base_vcores": 1024,
        "burst_vcores": 3072,
        "max_parallel_workers": 16,
        "max_repartition_gb": 2000.0,
        "optimize_warn_gb": 3000.0,
    },
    "F1024": {
        "base_vcores": 2048,
        "burst_vcores": 6144,
        "max_parallel_workers": 24,
        "max_repartition_gb": 4000.0,
        "optimize_warn_gb": 6000.0,
    },
    "F2048": {
        "base_vcores": 4096,
        "burst_vcores": 12288,
        "max_parallel_workers": 32,
        "max_repartition_gb": 8000.0,
        "optimize_warn_gb": 12000.0,
    },
}

LOG_TABLE_NAME = "_maintenance_log"

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("table_path", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("status", StringType(), False),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("num_files_before", LongType(), True),
    StructField("num_files_after", LongType(), True),
    StructField("size_bytes", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("dry_run", BooleanType(), False),
])


# ---------------------------------------------------------------------------
# Data classes
# ---------------------------------------------------------------------------

@dataclass
class MaintenanceConfig:
    """Configuration for the maintenance run."""
    lakehouse_abfss_path: str
    log_lakehouse_abfss_path: str
    sku: str = "F64"  # "F32" | "F64" | "F128"

    exclude_tables: list[str] = field(default_factory=list)
    partition_columns: dict[str, list[str]] = field(default_factory=dict)

    enable_repartition: bool = False
    repartition_size_limit_gb: float = DEFAULT_REPARTITION_SIZE_LIMIT_GB

    enable_vorder: bool = False
    vorder_tables: Optional[list[str]] = None

    vacuum_retention_hours: int = 168
    execution_mode: str = "sequential"   # "sequential" | "parallel"
    max_workers: int = 4
    continue_on_error: bool = True
    enable_logging: bool = True
    dry_run: bool = False
    collect_file_metrics: bool = False


@dataclass
class TableInfo:
    """Metadata for a discovered Delta table."""
    name: str
    path: str
    size_bytes: int = 0
    num_files: int = 0
    partition_columns: list[str] = field(default_factory=list)
    properties: dict = field(default_factory=dict)


@dataclass
class OperationResult:
    """Result of a single maintenance operation on one table."""
    table_name: str
    table_path: str
    operation: str
    status: str            # SUCCESS | FAILED | SKIPPED | DRY_RUN
    start_time: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    end_time: Optional[datetime] = None
    duration_seconds: float = 0.0
    num_files_before: Optional[int] = None
    num_files_after: Optional[int] = None
    size_bytes: Optional[int] = None
    error_message: Optional[str] = None
    dry_run: bool = False


# ---------------------------------------------------------------------------
# Logging setup
# ---------------------------------------------------------------------------

def _get_logger() -> logging.Logger:
    logger = logging.getLogger("fabric_delta_maintenance")
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(
            logging.Formatter(
                "%(asctime)s [%(levelname)s] %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S"
            )
        )
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    return logger

logger = _get_logger()


# ---------------------------------------------------------------------------
# Input validation
# ---------------------------------------------------------------------------

def _detect_managed_artifact(abfss_path: str) -> Optional[str]:
    """
    Return the managed Fabric artifact type embedded in an ABFSS path, or None.

    Matches segments such as ".Warehouse", ".MirroredDatabase", ".KQLDatabase",
    and ".SQLDatabase" whose Delta tables are system-managed by Fabric. Detection
    is case-insensitive and handles both "{name}.Warehouse/Tables/..." and a
    bare "{name}.Warehouse" suffix with no trailing slash.
    """
    lowered = abfss_path.lower().rstrip("/")
    for suffix in MANAGED_ARTIFACT_SUFFIXES:
        if lowered.endswith(suffix) or f"{suffix}/" in lowered:
            return suffix.lstrip(".")
    return None


def _validate_lakehouse_path(abfss_path: str, field_name: str) -> None:
    """
    Reject ABFSS paths that target system-managed Fabric artifacts.

    OPTIMIZE, VACUUM, and repartitioning are user-driven operations that
    require the caller to own the Delta layout. Warehouse, MirroredDatabase,
    KQLDatabase, and SQLDatabase artifacts are managed by Fabric; running
    maintenance against their paths can corrupt catalog state. Paths missing
    the ".Lakehouse" segment are flagged with a warning but allowed, since
    custom mounts and future artifact types may legitimately surface here.
    """
    if not abfss_path or not abfss_path.startswith("abfss://"):
        raise ValueError(
            f"{field_name} must be a valid ABFSS URI. Got: {abfss_path}"
        )
    artifact = _detect_managed_artifact(abfss_path)
    if artifact is not None:
        raise ValueError(
            f"{field_name} points at a system-managed Fabric artifact "
            f"({artifact}). This module supports Lakehouse paths only; "
            f"Warehouse, MirroredDatabase, KQLDatabase, and SQLDatabase "
            f"tables are managed by Fabric and must not be modified by "
            f"user-driven OPTIMIZE/VACUUM/repartitioning. Got: {abfss_path}"
        )
    if LAKEHOUSE_ARTIFACT_SUFFIX not in abfss_path.lower():
        logger.warning(
            "%s does not contain a '.Lakehouse' segment; proceeding, but "
            "verify this path resolves to a Lakehouse-managed location. "
            "Path: %s",
            field_name, abfss_path,
        )


def _validate_config(config: MaintenanceConfig) -> None:
    """
    Validate configuration and apply SKU-specific guardrails.

    Guardrails silently clamp values to safe ceilings rather than
    rejecting; all overrides are logged as warnings so the operator
    knows what changed.
    """
    _validate_lakehouse_path(config.lakehouse_abfss_path, "lakehouse_abfss_path")
    _validate_lakehouse_path(config.log_lakehouse_abfss_path, "log_lakehouse_abfss_path")

    # --- SKU validation ---
    sku_upper = config.sku.upper().strip()
    if sku_upper not in SUPPORTED_SKUS:
        raise ValueError(
            f"sku must be one of {sorted(SUPPORTED_SKUS)}. Got: '{config.sku}'"
        )
    config.sku = sku_upper
    profile = SKU_PROFILES[sku_upper]

    # --- Apply SKU guardrails ---

    # max_workers: clamp to SKU ceiling
    sku_max_workers = profile["max_parallel_workers"]
    if config.max_workers > sku_max_workers:
        logger.warning(
            "SKU %s: max_workers=%d exceeds safe ceiling of %d. Clamping to %d.",
            sku_upper, config.max_workers, sku_max_workers, sku_max_workers,
        )
        config.max_workers = sku_max_workers

    # repartition_size_limit_gb: clamp to SKU ceiling
    sku_max_repart = profile["max_repartition_gb"]
    if config.repartition_size_limit_gb > sku_max_repart:
        logger.warning(
            "SKU %s: repartition_size_limit_gb=%.0f exceeds safe ceiling of %.0f GB. "
            "Clamping to %.0f GB.",
            sku_upper, config.repartition_size_limit_gb,
            sku_max_repart, sku_max_repart,
        )
        config.repartition_size_limit_gb = sku_max_repart

    # parallel mode on F32: warn about contention risk
    if sku_upper == "F32" and config.execution_mode == "parallel":
        logger.warning(
            "SKU F32: parallel mode with 64 base vCores has limited headroom. "
            "Workers capped at %d. Consider sequential for tables > 10 GB.",
            config.max_workers,
        )

    if config.execution_mode not in ("sequential", "parallel"):
        raise ValueError(
            f"execution_mode must be 'sequential' or 'parallel'. Got: {config.execution_mode}"
        )

    if config.max_workers < 1:
        raise ValueError(f"max_workers must be >= 1. Got: {config.max_workers}")

    # Enforce vacuum retention floor
    if config.vacuum_retention_hours < MIN_VACUUM_RETENTION_HOURS:
        logger.warning(
            "vacuum_retention_hours=%d is below the minimum of %d. "
            "Overriding to %d hours (7 days).",
            config.vacuum_retention_hours,
            MIN_VACUUM_RETENTION_HOURS,
            MIN_VACUUM_RETENTION_HOURS,
        )
        config.vacuum_retention_hours = MIN_VACUUM_RETENTION_HOURS

    if config.repartition_size_limit_gb <= 0:
        raise ValueError(
            f"repartition_size_limit_gb must be > 0. Got: {config.repartition_size_limit_gb}"
        )

    # Validate partition_columns keys do not overlap with exclude_tables
    excluded_set = set(config.exclude_tables)
    for tbl in config.partition_columns:
        if tbl in excluded_set:
            logger.warning(
                "Table '%s' is in both partition_columns and exclude_tables; it will be skipped.",
                tbl,
            )


# ---------------------------------------------------------------------------
# Hadoop FileSystem helpers
# ---------------------------------------------------------------------------

def _get_hadoop_fs(spark: SparkSession, abfss_path: str):
    """Return a Hadoop FileSystem handle for the given ABFSS URI."""
    hadoop_conf = spark._jsc.hadoopConfiguration()
    uri = spark._jvm.java.net.URI(abfss_path)
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, hadoop_conf)


def _path_exists(spark: SparkSession, fs, path_str: str) -> bool:
    """Check whether a path exists on the Hadoop filesystem."""
    return fs.exists(spark._jvm.org.apache.hadoop.fs.Path(path_str))


def _list_subdirs(spark: SparkSession, fs, parent_path: str) -> list[str]:
    """List immediate subdirectory paths under parent_path."""
    parent = spark._jvm.org.apache.hadoop.fs.Path(parent_path)
    if not fs.exists(parent):
        return []
    statuses = fs.listStatus(parent)
    return [
        str(s.getPath().toString())
        for s in statuses
        if s.isDirectory()
    ]


# ---------------------------------------------------------------------------
# Table discovery
# ---------------------------------------------------------------------------

def _discover_tables(
    spark: SparkSession,
    config: MaintenanceConfig,
) -> list[TableInfo]:
    """
    Discover Delta tables under {lakehouse_abfss_path}/Tables.

    Handles both schema-enabled lakehouses (Tables/schema/table)
    and flat lakehouses (Tables/table). A directory qualifies as a
    Delta table only if it contains a _delta_log subdirectory.
    """
    base_path = config.lakehouse_abfss_path.rstrip("/")
    tables_root = f"{base_path}/Tables"
    fs = _get_hadoop_fs(spark, base_path)
    exclude_set = {t.lower() for t in config.exclude_tables}

    discovered: list[TableInfo] = []

    top_level_dirs = _list_subdirs(spark, fs, tables_root)

    for dir_path in top_level_dirs:
        dir_name = dir_path.rstrip("/").split("/")[-1]

        # Skip hidden/system directories
        if dir_name.startswith("_") or dir_name.startswith("."):
            continue

        delta_log = f"{dir_path}/_delta_log"
        if _path_exists(spark, fs, delta_log):
            # Direct table (flat lakehouse)
            if dir_name.lower() not in exclude_set:
                discovered.append(TableInfo(name=dir_name, path=dir_path))
        else:
            # Could be a schema directory; check children
            child_dirs = _list_subdirs(spark, fs, dir_path)
            for child_path in child_dirs:
                child_name = child_path.rstrip("/").split("/")[-1]
                if child_name.startswith("_") or child_name.startswith("."):
                    continue
                child_delta_log = f"{child_path}/_delta_log"
                if _path_exists(spark, fs, child_delta_log):
                    # Schema-qualified name: schema.table
                    qualified_name = f"{dir_name}.{child_name}"
                    if (
                        qualified_name.lower() not in exclude_set
                        and child_name.lower() not in exclude_set
                    ):
                        discovered.append(
                            TableInfo(name=qualified_name, path=child_path)
                        )

    logger.info("Discovered %d Delta table(s).", len(discovered))
    return discovered


def _enrich_table_info(
    spark: SparkSession,
    table: TableInfo,
) -> TableInfo:
    """
    Populate size_bytes, num_files, partition_columns, and properties
    from DESCRIBE DETAIL.
    """
    try:
        detail_df = spark.sql(f"DESCRIBE DETAIL delta.`{table.path}`")
        row = detail_df.first()
        if row:
            table.size_bytes = row["sizeInBytes"] or 0
            table.num_files = row["numFiles"] or 0
            table.partition_columns = list(row["partitionColumns"] or [])
            table.properties = dict(row["properties"] or {})
    except Exception as exc:
        logger.warning(
            "Could not retrieve DESCRIBE DETAIL for '%s': %s",
            table.name, str(exc)
        )
    return table


# ---------------------------------------------------------------------------
# Concurrency auto-throttle
# ---------------------------------------------------------------------------

def _calculate_effective_max_workers(
    tables: list[TableInfo],
    requested_max_workers: int,
    sku: str = "F64",
) -> int:
    """
    Reduce concurrency based on table sizes and SKU capacity.

    SKU-aware thresholds:
      F32:  "large" = 5 GB,  "very large" = 25 GB
      F64:  "large" = 10 GB, "very large" = 50 GB
      F128: "large" = 20 GB, "very large" = 100 GB

    Heuristic:
      - Any table > "very large" threshold: cap at 1 (F32) or 2
      - > 50% of tables > "large" threshold: cap at 2
      - > 25% of tables > "large" threshold: cap at 3
      - Otherwise: use requested max_workers (already SKU-clamped)
    """
    if not tables:
        return 1

    profile = SKU_PROFILES.get(sku, SKU_PROFILES["F64"])

    # Scale thresholds relative to SKU capacity
    # F32 has half the vCores of F64, so thresholds are halved
    sku_factor = profile["base_vcores"] / 128  # normalized to F64 = 1.0
    large_threshold = int(LARGE_TABLE_THRESHOLD_BYTES * sku_factor)
    very_large_threshold = int(VERY_LARGE_TABLE_THRESHOLD_BYTES * sku_factor)

    very_large = sum(1 for t in tables if t.size_bytes > very_large_threshold)
    if very_large > 0:
        # F32 with very large tables: single worker is safest
        cap = 1 if sku == "F32" else 2
        effective = min(requested_max_workers, cap)
        logger.info(
            "Auto-throttle [%s]: %d table(s) > %d GB detected; capping workers at %d.",
            sku, very_large, very_large_threshold // (1024**3), effective,
        )
        return effective

    large = sum(1 for t in tables if t.size_bytes > large_threshold)
    large_ratio = large / len(tables)

    if large_ratio > 0.5:
        effective = min(requested_max_workers, 2)
    elif large_ratio > 0.25:
        effective = min(requested_max_workers, 3)
    else:
        effective = requested_max_workers

    if effective != requested_max_workers:
        logger.info(
            "Auto-throttle [%s]: %d/%d tables > %d GB (%.0f%%); capping workers at %d.",
            sku, large, len(tables), large_threshold // (1024**3),
            large_ratio * 100, effective,
        )
    return effective


# ---------------------------------------------------------------------------
# Maintenance operations
# ---------------------------------------------------------------------------

def _collect_file_count(spark: SparkSession, table_path: str) -> Optional[int]:
    """Return numFiles from DESCRIBE DETAIL; None on failure."""
    try:
        row = spark.sql(f"DESCRIBE DETAIL delta.`{table_path}`").first()
        return row["numFiles"] if row else None
    except Exception:
        return None


def _repartition_table(
    spark: SparkSession,
    table: TableInfo,
    target_partition_cols: list[str],
    config: MaintenanceConfig,
    run_id: str,
) -> OperationResult:
    """
    Repartition a table using the staging-table pattern:
      1. Read existing data into DataFrame
      2. Count source rows
      3. Write to staging path with partitionBy
      4. Validate staging row count matches source
      5. Delete original table directory
      6. Rename staging to original path
      7. Verify final row count

    Gated by enable_repartition and size limit.
    """
    result = OperationResult(
        table_name=table.name,
        table_path=table.path,
        operation="REPARTITION",
        dry_run=config.dry_run,
    )
    result.start_time = datetime.now(timezone.utc)

    # Guard: already partitioned with same columns?
    if sorted(table.partition_columns) == sorted(target_partition_cols):
        result.status = "SKIPPED"
        result.end_time = datetime.now(timezone.utc)
        result.duration_seconds = 0.0
        logger.info(
            "[%s] REPARTITION skipped: already partitioned by %s.",
            table.name, target_partition_cols,
        )
        return result

    # Guard: repartition not enabled
    if not config.enable_repartition:
        result.status = "SKIPPED"
        result.end_time = datetime.now(timezone.utc)
        logger.info(
            "[%s] REPARTITION skipped: enable_repartition is False.",
            table.name,
        )
        return result

    # Guard: table too large for safe repartition
    size_gb = table.size_bytes / (1024**3)
    if size_gb > config.repartition_size_limit_gb:
        result.status = "SKIPPED"
        result.end_time = datetime.now(timezone.utc)
        result.error_message = (
            f"Table size {size_gb:.1f} GB exceeds repartition limit "
            f"of {config.repartition_size_limit_gb} GB."
        )
        logger.warning("[%s] REPARTITION skipped: %s", table.name, result.error_message)
        return result

    if config.dry_run:
        result.status = "DRY_RUN"
        result.end_time = datetime.now(timezone.utc)
        logger.info(
            "[%s] REPARTITION dry run: would repartition by %s (%.1f GB).",
            table.name, target_partition_cols, size_gb,
        )
        return result

    staging_path = f"{table.path}__staging_{run_id[:8]}"
    fs = _get_hadoop_fs(spark, config.lakehouse_abfss_path)

    try:
        # Step 1: Read and count source
        source_df = spark.read.format("delta").load(table.path)
        source_schema = source_df.schema
        source_count = source_df.count()
        logger.info(
            "[%s] REPARTITION source count: %d rows, %.1f GB.",
            table.name, source_count, size_gb,
        )

        # Validate partition columns exist in schema
        schema_fields = {f.name.lower() for f in source_schema.fields}
        missing = [c for c in target_partition_cols if c.lower() not in schema_fields]
        if missing:
            raise ValueError(
                f"Partition column(s) {missing} not found in table schema. "
                f"Available: {sorted(schema_fields)}"
            )

        # Step 2: Write to staging with partitionBy
        (
            source_df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy(*target_partition_cols)
            .save(staging_path)
        )

        # Step 3: Validate staging row count
        staging_count = spark.read.format("delta").load(staging_path).count()
        if staging_count != source_count:
            raise ValueError(
                f"Row count mismatch after repartition: "
                f"source={source_count}, staging={staging_count}. "
                f"Aborting; staging left at {staging_path} for inspection."
            )

        # Step 4: Remove original, rename staging to original
        original_hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(table.path)
        staging_hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(staging_path)

        fs.delete(original_hadoop_path, True)  # recursive delete
        rename_success = fs.rename(staging_hadoop_path, original_hadoop_path)
        if not rename_success:
            raise RuntimeError(
                f"Hadoop rename from {staging_path} to {table.path} returned False. "
                f"Data may be at staging path; manual recovery required."
            )

        # Step 5: Final row count verification
        final_count = spark.read.format("delta").load(table.path).count()
        if final_count != source_count:
            raise ValueError(
                f"Post-rename row count mismatch: source={source_count}, final={final_count}."
            )

        result.status = "SUCCESS"
        logger.info(
            "[%s] REPARTITION completed: %d rows, partitioned by %s.",
            table.name, final_count, target_partition_cols,
        )

    except Exception as exc:
        result.status = "FAILED"
        result.error_message = str(exc)
        logger.error("[%s] REPARTITION failed: %s", table.name, exc)
        # Attempt cleanup of staging if it exists
        try:
            staging_hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(staging_path)
            if fs.exists(staging_hadoop_path):
                logger.warning(
                    "[%s] Staging directory left at %s for manual review.",
                    table.name, staging_path,
                )
        except Exception:
            pass

    result.end_time = datetime.now(timezone.utc)
    result.duration_seconds = (result.end_time - result.start_time).total_seconds()
    return result


def _optimize_table(
    spark: SparkSession,
    table: TableInfo,
    config: MaintenanceConfig,
) -> OperationResult:
    """Run OPTIMIZE (compaction) on a Delta table via path."""
    result = OperationResult(
        table_name=table.name,
        table_path=table.path,
        operation="OPTIMIZE",
        dry_run=config.dry_run,
    )
    result.start_time = datetime.now(timezone.utc)

    if config.collect_file_metrics:
        result.num_files_before = _collect_file_count(spark, table.path)
        result.size_bytes = table.size_bytes

    # SKU-aware size warning (not a block, just observability)
    warn_gb = SKU_PROFILES.get(config.sku, {}).get("optimize_warn_gb", 300.0)
    table_gb = table.size_bytes / (1024**3)
    if table_gb > warn_gb:
        logger.warning(
            "[%s] OPTIMIZE: table is %.1f GB, exceeding %s advisory threshold "
            "of %.0f GB. Expect elevated CU consumption and longer runtime.",
            table.name, table_gb, config.sku, warn_gb,
        )

    if config.dry_run:
        result.status = "DRY_RUN"
        result.end_time = datetime.now(timezone.utc)
        logger.info(
            "[%s] OPTIMIZE dry run: %d files, %d bytes.",
            table.name, table.num_files, table.size_bytes,
        )
        return result

    try:
        spark.sql(f"OPTIMIZE delta.`{table.path}`")
        result.status = "SUCCESS"

        if config.collect_file_metrics:
            result.num_files_after = _collect_file_count(spark, table.path)

        logger.info("[%s] OPTIMIZE completed.", table.name)

    except Exception as exc:
        result.status = "FAILED"
        result.error_message = str(exc)
        logger.error("[%s] OPTIMIZE failed: %s", table.name, exc)

    result.end_time = datetime.now(timezone.utc)
    result.duration_seconds = (result.end_time - result.start_time).total_seconds()
    return result


def _apply_vorder(
    spark: SparkSession,
    table: TableInfo,
    config: MaintenanceConfig,
) -> OperationResult:
    """
    Apply V-Order optimization via OPTIMIZE ... VORDER.

    Runtime 1.3 supports the VORDER keyword directly in OPTIMIZE.
    This sets the V-Order layout during compaction in a single pass,
    which is more efficient than setting the table property separately
    and then running a second OPTIMIZE.

    Collects num_files_before/num_files_after when collect_file_metrics
    is enabled, since this operation performs compaction (OPTIMIZE VORDER)
    and the file count delta is meaningful.

    Only runs if:
      - enable_vorder is True
      - vorder_tables is None (apply to all) OR table is in vorder_tables
    """
    result = OperationResult(
        table_name=table.name,
        table_path=table.path,
        operation="VORDER",
        dry_run=config.dry_run,
    )
    result.start_time = datetime.now(timezone.utc)

    # Check eligibility
    if not config.enable_vorder:
        result.status = "SKIPPED"
        result.end_time = datetime.now(timezone.utc)
        return result

    if config.vorder_tables is not None:
        eligible = {t.lower() for t in config.vorder_tables}
        # Match on full name or just the table portion (for schema-qualified names)
        table_name_lower = table.name.lower()
        short_name_lower = table.name.split(".")[-1].lower()
        if table_name_lower not in eligible and short_name_lower not in eligible:
            result.status = "SKIPPED"
            result.end_time = datetime.now(timezone.utc)
            logger.info(
                "[%s] VORDER skipped: table not in vorder_tables list.", table.name,
            )
            return result

    if config.collect_file_metrics:
        result.num_files_before = _collect_file_count(spark, table.path)
        result.size_bytes = table.size_bytes

    if config.dry_run:
        result.status = "DRY_RUN"
        result.end_time = datetime.now(timezone.utc)
        logger.info("[%s] VORDER dry run.", table.name)
        return result

    try:
        # Set the table property for persistent V-Order on future writes,
        # then run OPTIMIZE VORDER for the current compaction pass.
        spark.sql(
            f"ALTER TABLE delta.`{table.path}` "
            f"SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')"
        )
        spark.sql(f"OPTIMIZE delta.`{table.path}` VORDER")
        result.status = "SUCCESS"

        if config.collect_file_metrics:
            result.num_files_after = _collect_file_count(spark, table.path)

        logger.info("[%s] VORDER applied (table property set + OPTIMIZE VORDER).", table.name)

    except Exception as exc:
        result.status = "FAILED"
        result.error_message = str(exc)
        logger.error("[%s] VORDER failed: %s", table.name, exc)

    result.end_time = datetime.now(timezone.utc)
    result.duration_seconds = (result.end_time - result.start_time).total_seconds()
    return result


def _vacuum_table(
    spark: SparkSession,
    table: TableInfo,
    config: MaintenanceConfig,
) -> OperationResult:
    """Run VACUUM with enforced minimum retention."""
    result = OperationResult(
        table_name=table.name,
        table_path=table.path,
        operation="VACUUM",
        dry_run=config.dry_run,
    )
    result.start_time = datetime.now(timezone.utc)

    retention = max(config.vacuum_retention_hours, MIN_VACUUM_RETENTION_HOURS)

    if config.dry_run:
        result.status = "DRY_RUN"
        result.end_time = datetime.now(timezone.utc)
        logger.info(
            "[%s] VACUUM dry run: retention=%d hours.", table.name, retention,
        )
        return result

    try:
        spark.sql(f"VACUUM delta.`{table.path}` RETAIN {retention} HOURS")
        result.status = "SUCCESS"
        logger.info("[%s] VACUUM completed (retention=%d hours).", table.name, retention)

    except Exception as exc:
        result.status = "FAILED"
        result.error_message = str(exc)
        logger.error("[%s] VACUUM failed: %s", table.name, exc)

    result.end_time = datetime.now(timezone.utc)
    result.duration_seconds = (result.end_time - result.start_time).total_seconds()
    return result


# ---------------------------------------------------------------------------
# Per-table orchestrator
# ---------------------------------------------------------------------------

def _maintain_single_table(
    spark: SparkSession,
    table: TableInfo,
    config: MaintenanceConfig,
    run_id: str,
) -> list[OperationResult]:
    """
    Execute the full maintenance sequence for one table:
      1. Repartition (conditional)
      2. OPTIMIZE (always, unless V-Order is eligible)
      3. V-Order (conditional; runs its own OPTIMIZE VORDER)
         - If VORDER fails, falls back to plain OPTIMIZE so the
           table still gets compacted regardless.
      4. VACUUM (always)

    When V-Order is applied, it runs OPTIMIZE VORDER, which combines
    compaction with V-Order in a single pass. To avoid a redundant
    double-OPTIMIZE, the plain OPTIMIZE step is skipped for V-Order
    eligible tables.
    """
    results: list[OperationResult] = []

    # --- 1. Repartition (if configured for this table) ---
    target_name = table.name
    short_name = table.name.split(".")[-1]
    target_partitions = (
        config.partition_columns.get(target_name)
        or config.partition_columns.get(short_name)
    )
    if target_partitions:
        repartition_result = _repartition_table(
            spark, table, target_partitions, config, run_id
        )
        results.append(repartition_result)
        if repartition_result.status == "FAILED" and not config.continue_on_error:
            return results
        # Refresh table info after repartition (partition cols may have changed)
        if repartition_result.status == "SUCCESS":
            _enrich_table_info(spark, table)

    # --- Determine if V-Order will run (to avoid double OPTIMIZE) ---
    vorder_will_run = False
    if config.enable_vorder:
        if config.vorder_tables is None:
            vorder_will_run = True
        else:
            eligible = {t.lower() for t in config.vorder_tables}
            vorder_will_run = (
                table.name.lower() in eligible
                or short_name.lower() in eligible
            )

    # --- 2. OPTIMIZE (skip if V-Order will handle compaction) ---
    if vorder_will_run:
        logger.info(
            "[%s] Skipping plain OPTIMIZE; VORDER step will run OPTIMIZE VORDER.",
            table.name,
        )
    else:
        optimize_result = _optimize_table(spark, table, config)
        results.append(optimize_result)
        if optimize_result.status == "FAILED" and not config.continue_on_error:
            return results

    # --- 3. V-Order ---
    vorder_result = _apply_vorder(spark, table, config)
    results.append(vorder_result)

    # Fallback: if VORDER failed and we skipped plain OPTIMIZE earlier,
    # the table has received no compaction at all. Run plain OPTIMIZE
    # as a recovery step so compaction still happens.
    fallback_succeeded = False
    if vorder_result.status == "FAILED" and vorder_will_run:
        logger.warning(
            "[%s] VORDER failed; falling back to plain OPTIMIZE for compaction.",
            table.name,
        )
        fallback_result = _optimize_table(spark, table, config)
        fallback_result.operation = "OPTIMIZE_FALLBACK"
        results.append(fallback_result)
        fallback_succeeded = (fallback_result.status == "SUCCESS")
        if fallback_result.status == "FAILED" and not config.continue_on_error:
            return results

    # Only abort on VORDER failure if continue_on_error=False AND
    # no successful fallback rescued compaction.
    if (
        vorder_result.status == "FAILED"
        and not config.continue_on_error
        and not fallback_succeeded
    ):
        return results

    # --- 4. VACUUM ---
    vacuum_result = _vacuum_table(spark, table, config)
    results.append(vacuum_result)

    return results


# ---------------------------------------------------------------------------
# Log persistence
# ---------------------------------------------------------------------------

def _write_log_to_delta(
    spark: SparkSession,
    results: list[OperationResult],
    config: MaintenanceConfig,
    run_id: str,
) -> None:
    """
    Append operation results to the maintenance log Delta table
    in the separate log Lakehouse.
    """
    if not config.enable_logging:
        return

    log_path = (
        f"{config.log_lakehouse_abfss_path.rstrip('/')}/Tables/{LOG_TABLE_NAME}"
    )

    rows = []
    for r in results:
        rows.append((
            run_id,
            r.table_name,
            r.table_path,
            r.operation,
            r.status,
            r.start_time,
            r.end_time,
            r.duration_seconds,
            r.num_files_before,
            r.num_files_after,
            r.size_bytes,
            r.error_message,
            r.dry_run,
        ))

    if not rows:
        logger.info("No results to log.")
        return

    log_df = spark.createDataFrame(rows, schema=LOG_SCHEMA)

    try:
        (
            log_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .save(log_path)
        )
        logger.info(
            "Wrote %d log record(s) to %s.", len(rows), log_path,
        )
    except Exception as exc:
        logger.error("Failed to write maintenance log: %s", exc)


# ---------------------------------------------------------------------------
# Summary builder
# ---------------------------------------------------------------------------

def _build_summary(
    results: list[OperationResult],
    tables: list[TableInfo],
    config: MaintenanceConfig,
    run_id: str,
    total_duration: float,
) -> dict:
    """Build a structured summary of the maintenance run."""
    succeeded = set()
    failed = set()
    skipped = set()
    dry_run_tables = set()

    for r in results:
        if r.status == "SUCCESS":
            succeeded.add(r.table_name)
        elif r.status == "FAILED":
            failed.add(r.table_name)
        elif r.status == "DRY_RUN":
            dry_run_tables.add(r.table_name)
        # SKIPPED operations don't mean the table itself was skipped;
        # a table is "skipped" only if ALL its operations were SKIPPED.

    # Tables where every operation was SKIPPED
    from collections import defaultdict
    table_ops = defaultdict(list)
    for r in results:
        table_ops[r.table_name].append(r.status)
    for tname, statuses in table_ops.items():
        if all(s == "SKIPPED" for s in statuses):
            skipped.add(tname)

    errors = [
        {"table": r.table_name, "operation": r.operation, "error": r.error_message}
        for r in results
        if r.status == "FAILED"
    ]

    summary = {
        "run_id": run_id,
        "execution_mode": config.execution_mode,
        "dry_run": config.dry_run,
        "total_duration_seconds": round(total_duration, 2),
        "tables_discovered": len(tables),
        "tables_succeeded": sorted(succeeded - failed),
        "tables_failed": sorted(failed),
        "tables_skipped": sorted(skipped),
        "tables_dry_run": sorted(dry_run_tables),
        "total_operations": len(results),
        "operation_breakdown": {
            "SUCCESS": sum(1 for r in results if r.status == "SUCCESS"),
            "FAILED": sum(1 for r in results if r.status == "FAILED"),
            "SKIPPED": sum(1 for r in results if r.status == "SKIPPED"),
            "DRY_RUN": sum(1 for r in results if r.status == "DRY_RUN"),
        },
        "errors": errors,
    }
    return summary


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def run_maintenance(
    spark: SparkSession,
    config: MaintenanceConfig,
) -> dict:
    """
    Execute automated Delta table maintenance across all tables
    in the target Lakehouse.

    Args:
        spark:  Active SparkSession (pre-built in Fabric notebooks).
        config: MaintenanceConfig with all settings.

    Returns:
        dict: Structured summary with succeeded/failed/skipped tables,
              operation counts, errors, and timing.

    Raises:
        ValueError: On invalid configuration.
    """
    run_id = str(uuid.uuid4())
    run_start = time.time()

    logger.info("=" * 70)
    logger.info("FABRIC DELTA MAINTENANCE  |  Run ID: %s", run_id)
    logger.info("SKU: %s  |  Lakehouse: %s", config.sku, config.lakehouse_abfss_path)
    logger.info("Log target: %s", config.log_lakehouse_abfss_path)
    logger.info("Mode: %s  |  Dry run: %s", config.execution_mode, config.dry_run)
    logger.info("=" * 70)

    # --- Validate ---
    _validate_config(config)

    # --- Discover tables ---
    tables = _discover_tables(spark, config)
    if not tables:
        logger.warning("No Delta tables found. Exiting.")
        return _build_summary([], [], config, run_id, time.time() - run_start)

    # --- Enrich with DESCRIBE DETAIL ---
    for table in tables:
        _enrich_table_info(spark, table)

    logger.info(
        "Table inventory: %s",
        ", ".join(
            f"{t.name} ({t.num_files} files, {t.size_bytes / (1024**3):.2f} GB)"
            for t in tables
        ),
    )

    # --- Execute maintenance ---
    all_results: list[OperationResult] = []

    if config.execution_mode == "sequential":
        for table in tables:
            try:
                table_results = _maintain_single_table(spark, table, config, run_id)
                all_results.extend(table_results)
            except Exception as exc:
                logger.error(
                    "[%s] Unhandled error during maintenance: %s\n%s",
                    table.name, exc, traceback.format_exc(),
                )
                all_results.append(OperationResult(
                    table_name=table.name,
                    table_path=table.path,
                    operation="MAINTENANCE",
                    status="FAILED",
                    error_message=str(exc),
                    start_time=datetime.now(timezone.utc),
                    end_time=datetime.now(timezone.utc),
                    dry_run=config.dry_run,
                ))
                if not config.continue_on_error:
                    logger.error("continue_on_error=False; aborting run.")
                    break

    elif config.execution_mode == "parallel":
        effective_workers = _calculate_effective_max_workers(
            tables, config.max_workers, config.sku,
        )
        logger.info(
            "Parallel execution: %d worker(s) (requested %d).",
            effective_workers, config.max_workers,
        )

        # ThreadPoolExecutor shares the Spark driver; each thread
        # submits Spark SQL jobs that are queued by the Spark scheduler.
        with ThreadPoolExecutor(max_workers=effective_workers) as executor:
            futures = {
                executor.submit(
                    _maintain_single_table, spark, table, config, run_id
                ): table
                for table in tables
            }

            for future in as_completed(futures):
                table = futures[future]
                try:
                    table_results = future.result()
                    all_results.extend(table_results)
                except Exception as exc:
                    logger.error(
                        "[%s] Unhandled error in parallel worker: %s\n%s",
                        table.name, exc, traceback.format_exc(),
                    )
                    all_results.append(OperationResult(
                        table_name=table.name,
                        table_path=table.path,
                        operation="MAINTENANCE",
                        status="FAILED",
                        error_message=str(exc),
                        start_time=datetime.now(timezone.utc),
                        end_time=datetime.now(timezone.utc),
                        dry_run=config.dry_run,
                    ))

    # --- Persist logs ---
    _write_log_to_delta(spark, all_results, config, run_id)

    # --- Summary ---
    total_duration = time.time() - run_start
    summary = _build_summary(all_results, tables, config, run_id, total_duration)

    logger.info("=" * 70)
    logger.info("MAINTENANCE COMPLETE  |  Duration: %.1f seconds", total_duration)
    logger.info(
        "Results: %d succeeded, %d failed, %d skipped",
        summary["operation_breakdown"]["SUCCESS"],
        summary["operation_breakdown"]["FAILED"],
        summary["operation_breakdown"]["SKIPPED"],
    )
    if summary["errors"]:
        logger.warning("Errors:")
        for err in summary["errors"]:
            logger.warning("  [%s] %s: %s", err["table"], err["operation"], err["error"])
    logger.info("=" * 70)

    return summary


In [ ]:
# ============================================================================
# CELL 2: PARAMETERS  (this cell is tagged "parameters")
# ============================================================================
#
# A Fabric Data Pipeline Notebook activity can override any of these
# variables via baseParameters. Example pipeline JSON:
#
#   "baseParameters": {
#     "dry_run":          { "type": "bool",   "value": false },
#     "sku":              { "type": "string", "value": "F128" },
#     "execution_mode":   { "type": "string", "value": "parallel" },
#     "max_workers":      { "type": "int",    "value": 8 }
#   }
#
# Structured per-table settings (partition_columns, vorder_tables,
# exclude_tables) live in Cell 3 because pipeline parameters are
# scalar-only.

lakehouse_abfss_path = (
    "abfss://<workspace-guid>@onelake.dfs.fabric.microsoft.com/<lakehouse-guid>"
)
log_lakehouse_abfss_path = (
    "abfss://<workspace-guid>@onelake.dfs.fabric.microsoft.com/<log-lakehouse-guid>"
)

sku = "F64"                       # F32 | F64 | F128 | F256 | F512 | F1024 | F2048
dry_run = True                    # Pipeline overrides to False for production
enable_vorder = True
enable_repartition = False        # Set True only after dry_run validation
execution_mode = "sequential"     # sequential | parallel
max_workers = 4
vacuum_retention_hours = 168      # 7 days (minimum enforced)
continue_on_error = True
collect_file_metrics = True


In [ ]:
# ============================================================================
# CELL 3: Configuration & Invocation
# ============================================================================
#
# Per-table settings (partition_columns, vorder_tables, exclude_tables) live
# here, not in the parameter cell. Edit them directly when tables are added
# or removed from the maintenance scope.

config = MaintenanceConfig(
    # --- From parameter cell ---
    lakehouse_abfss_path=lakehouse_abfss_path,
    log_lakehouse_abfss_path=log_lakehouse_abfss_path,
    sku=sku,
    dry_run=dry_run,
    enable_vorder=enable_vorder,
    enable_repartition=enable_repartition,
    execution_mode=execution_mode,
    max_workers=max_workers,
    vacuum_retention_hours=vacuum_retention_hours,
    continue_on_error=continue_on_error,
    collect_file_metrics=collect_file_metrics,

    # --- Static, edited in code ---
    exclude_tables=[
        "staging_temp",
        "dbo._maintenance_log",
    ],
    repartition_size_limit_gb=200.0,
    partition_columns={
        "fact_sales": ["sale_year", "sale_month"],
        "fact_events": ["event_date"],
    },
    vorder_tables=[
        "fact_sales",
        "dim_customer",
        "dim_product",
        "agg_daily_revenue",
    ],
    enable_logging=True,
)

summary = run_maintenance(spark, config)


In [ ]:
# ============================================================================
# CELL 4: Inspect Results
# ============================================================================

import json
print(json.dumps(summary, indent=2, default=str))


In [ ]:
# ============================================================================
# CELL 5: Query the Maintenance Log (after at least one real run)
# ============================================================================

log_path = f"{log_lakehouse_abfss_path}/Tables/_maintenance_log"

try:
    log_df = spark.read.format("delta").load(log_path)
    display(
        log_df
        .orderBy("start_time", ascending=False)
        .select(
            "run_id", "table_name", "operation", "status",
            "duration_seconds", "num_files_before", "num_files_after",
            "error_message", "dry_run"
        )
    )
except Exception as e:
    print(f"Log table not yet created (run with dry_run=False first): {e}")


In [ ]:
# ============================================================================
# CELL 6: Historical Analysis (optional)
# ============================================================================

# File compaction trend over time
try:
    log_df = spark.read.format("delta").load(log_path)

    from pyspark.sql.functions import col, avg, count

    compaction_trend = (
        log_df
        .filter(
            (col("operation") == "OPTIMIZE")
            & (col("status") == "SUCCESS")
            & col("num_files_before").isNotNull()
        )
        .groupBy("table_name")
        .agg(
            count("*").alias("optimize_runs"),
            avg("duration_seconds").alias("avg_duration_sec"),
            avg("num_files_before").alias("avg_files_before"),
            avg("num_files_after").alias("avg_files_after"),
        )
        .orderBy("avg_duration_sec", ascending=False)
    )
    display(compaction_trend)
except Exception:
    print("Insufficient log data for trend analysis.")


## Scheduling via Fabric Data Pipeline

1. Save this notebook in your Fabric workspace.
2. Confirm Cell 2 is tagged as the parameter cell. The "Parameters" badge must appear on the cell. (Open the cell toolbar's "..." menu and select **Toggle parameter cell** if the badge is missing.) Without this tag, pipeline parameters silently fail to override.
3. Create a Fabric Data Factory Pipeline.
4. Add a **Notebook** activity pointing at this notebook.
5. Open the activity's **Base parameters** section and add per-environment overrides:

   | Name | Type | Production value |
   |------|------|------------------|
   | `dry_run` | bool | `false` |
   | `sku` | string | matches your capacity (e.g. `F64`) |
   | `execution_mode` | string | `sequential` (or `parallel` on F64+) |
   | `max_workers` | int | matches the SKU profile ceiling |

6. Schedule the pipeline off-peak (recommended: daily 2:00 AM in the capacity's home time zone).
7. For staging vs production environments, create two pipelines pointing at the same notebook with different parameter values (e.g., the staging pipeline keeps `dry_run = true`).

### Parameters NOT exposed via pipeline

`exclude_tables`, `partition_columns`, and `vorder_tables` stay in Cell 3 because pipeline parameters are scalar-only. Edit Cell 3 directly when the table inventory changes.
